# Notebook 8: Support Vector Machines (SVM)
**DCS 404 — Data Science and Machine Learning**

---

## Learning Objectives
- Explain the concept of maximum-margin classification
- Understand support vectors and the decision hyperplane
- Describe the kernel trick for non-linear boundaries
- Apply SVM to classification and compare with other classifiers
- Tune the `C` and `gamma` hyperparameters

## 1. The Core Idea: Maximum Margin

Given linearly separable data, infinitely many hyperplanes can separate the two classes. Which one is best?

**SVM picks the hyperplane that maximises the margin** — the width of the empty "street" between the two classes.

- The data points that lie on the margin boundaries are called **support vectors** — they are the only points that "matter" for the decision boundary.
- A larger margin → better generalisation to unseen data.

$$\text{Decision hyperplane}: \mathbf{w}^T \mathbf{x} + b = 0$$
$$\text{Margin width} = \frac{2}{\|\mathbf{w}\|}$$

Maximising the margin is equivalent to minimising $\|\mathbf{w}\|$ subject to correct classification.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_classification, make_circles, make_moons, load_iris
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)

In [ ]:
# Generate linearly separable data and visualise the maximum-margin hyperplane
from sklearn.datasets import make_blobs
X_lin, y_lin = make_blobs(n_samples=60, centers=2, cluster_std=0.9, random_state=6)

svm_lin = SVC(kernel='linear', C=1e10)  # hard margin
svm_lin.fit(X_lin, y_lin)

def plot_svm_decision(model, X, y, ax, title):
    x_min, x_max = X[:,0].min()-1, X[:,0].max()+1
    y_min, y_max = X[:,1].min()-1, X[:,1].max()+1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=[-1e9, -1, 0, 1, 1e9],
                colors=['#FF8888','#FFCCCC','#CCFFCC','#88FF88'], alpha=0.4)
    ax.contour(xx, yy, Z, levels=[-1, 0, 1], linestyles=['--','-','--'],
               colors='black', linewidths=[1, 2, 1])
    ax.scatter(X[:,0], X[:,1], c=y, cmap='bwr', edgecolors='white', s=50)
    # Highlight support vectors
    sv = model.support_vectors_
    ax.scatter(sv[:,0], sv[:,1], s=200, facecolors='none', edgecolors='black',
               linewidths=2, label=f'Support vectors ({len(sv)})')
    ax.set_title(title)
    ax.legend(fontsize=8)

fig, ax = plt.subplots(figsize=(7, 5))
plot_svm_decision(svm_lin, X_lin, y_lin, ax, 'SVM: Maximum Margin (Linear Kernel)')
plt.tight_layout()
plt.show()

## 2. Soft Margin SVM — The `C` Parameter

Real data is rarely perfectly linearly separable. **Soft Margin SVM** allows some misclassifications:

$$\min_{\mathbf{w}, b} \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_i \xi_i$$

- $\xi_i \geq 0$ are **slack variables** allowing constraint violations
- **Large C**: fewer violations allowed → smaller margin, risk of overfitting
- **Small C**: more violations allowed → larger margin, risk of underfitting

In [ ]:
X_noisy, y_noisy = make_blobs(n_samples=100, centers=2, cluster_std=2.0, random_state=6)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, C_val in zip(axes, [0.01, 1.0, 100]):
    svm = SVC(kernel='linear', C=C_val)
    svm.fit(X_noisy, y_noisy)
    plot_svm_decision(svm, X_noisy, y_noisy, ax, f'C = {C_val}')
plt.suptitle('Effect of C on Soft Margin SVM')
plt.tight_layout()
plt.show()

## 3. The Kernel Trick for Non-linear Data

Many real datasets are **not linearly separable** in their original feature space.

**Idea**: Map data to a higher-dimensional space where it becomes linearly separable.

The **kernel trick** computes inner products in the high-dimensional space *without explicitly computing the transformation* — making it computationally feasible.

| Kernel | Formula | Use case |
|--------|---------|----------|
| **Linear** | $\mathbf{x}^T \mathbf{x}'$ | Linearly separable; high-dim sparse text |
| **RBF (Gaussian)** | $\exp(-\gamma \|\mathbf{x}-\mathbf{x}'\|^2)$ | Most common; smooth boundaries |
| **Polynomial** | $(\mathbf{x}^T \mathbf{x}' + r)^d$ | Polynomial features implicitly |

In [ ]:
X_circles, y_circles = make_circles(n_samples=200, noise=0.1, factor=0.3, random_state=42)
X_moons,   y_moons   = make_moons(n_samples=200, noise=0.15, random_state=42)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
datasets = [(X_circles, y_circles, 'Circles'), (X_moons, y_moons, 'Moons')]
kernels  = ['linear', 'rbf', 'poly']

for row, (X_d, y_d, dname) in enumerate(datasets):
    for col, kern in enumerate(kernels):
        ax = axes[row][col]
        svm_k = SVC(kernel=kern, C=1.0, gamma='scale')
        svm_k.fit(X_d, y_d)
        x_min, x_max = X_d[:,0].min()-0.5, X_d[:,0].max()+0.5
        y_min, y_max = X_d[:,1].min()-0.5, X_d[:,1].max()+0.5
        xx, yy = np.meshgrid(np.linspace(x_min,x_max,200), np.linspace(y_min,y_max,200))
        Z = svm_k.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        ax.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
        ax.scatter(X_d[:,0], X_d[:,1], c=y_d, cmap='bwr', edgecolors='white', s=25)
        acc = svm_k.score(X_d, y_d)
        ax.set_title(f'{dname} + {kern} kernel\nAcc={acc:.2f}')
        ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

## 4. SVM on a Real Dataset and Hyperparameter Tuning

In [ ]:
from sklearn.datasets import load_breast_cancer
cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target

X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

# Grid search for best C and gamma
pipe = Pipeline([('scaler', StandardScaler()), ('svm', SVC(kernel='rbf', probability=True))])
param_grid = {'svm__C': [0.1, 1, 10], 'svm__gamma': ['scale', 0.01, 0.001]}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_tr, y_tr)

print(f'Best params : {grid.best_params_}')
print(f'CV accuracy : {grid.best_score_:.4f}')

best_model = grid.best_estimator_
print(f'Test accuracy: {best_model.score(X_te, y_te):.4f}')

In [ ]:
# Visualise grid search results
import pandas as pd
results = pd.DataFrame(grid.cv_results_)
pivot = results.pivot_table(values='mean_test_score',
                             index='param_svm__gamma',
                             columns='param_svm__C')
fig, ax = plt.subplots(figsize=(7, 4))
im = ax.imshow(pivot.values, cmap='YlOrRd', vmin=pivot.values.min(), vmax=pivot.values.max())
ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns)
ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index)
ax.set_xlabel('C'); ax.set_ylabel('gamma')
ax.set_title('Grid Search: Validation Accuracy')
plt.colorbar(im, ax=ax)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f'{pivot.values[i,j]:.3f}', ha='center', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## Exercises

1. Using the Iris dataset, train an SVM with an RBF kernel and visualise the decision boundaries for all three pairs of features.
2. What happens when you set `C=1e-6` vs `C=1e6` on a noisy dataset? Describe and plot the difference.
3. The `gamma` parameter controls the RBF kernel width. A very high `gamma` means each sample has a small region of influence — does this lead to overfitting or underfitting? Verify experimentally.
4. Compare SVM (RBF) with Logistic Regression on the breast cancer dataset in terms of accuracy, training time, and interpretability.

## Reflection Questions
- Why is feature scaling critical for SVM but less important for tree-based models?
- Support vectors define the decision boundary. If you remove non-support-vector training points, does the boundary change? Why?
- Why can't you easily interpret SVM coefficients the way you can with logistic regression?

---
**Next →** `09_knn_naive_bayes.ipynb`